In [1]:
import torch
import torch.nn as nn

d:\understanding-llms\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [3]:
sentence = "Life is short, eat desert first"

dc = { s:i for i,s in enumerate(sorted(sentence.replace(',', ' ').split()))}
dc

{'Life': 0, 'desert': 1, 'eat': 2, 'first': 3, 'is': 4, 'short': 5}

In [4]:
sentence_int = torch.tensor([dc[s] for s in sentence.replace(',', ' ').split()])
sentence_int

tensor([0, 4, 5, 2, 1, 3])

In [5]:
vocab_size = 50_000

torch.manual_seed(123)
embed = torch.nn.Embedding(vocab_size, 3)
embedded_sentence = embed(sentence_int).detach()

print(embedded_sentence)
print(embedded_sentence.shape)

tensor([[ 0.3374, -0.1778, -0.3035],
        [ 0.1794,  1.8951,  0.4954],
        [ 0.2692, -0.0770, -1.0205],
        [-0.2196, -0.3792,  0.7671],
        [-0.5880,  0.3486,  0.6603],
        [-1.1925,  0.6984, -1.4097]])
torch.Size([6, 3])


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out_kq, d_out_v, num_heads):
        super().__init__()
        self.d_out_kq = d_out_kq
        self.d_out_v = d_out_v
        self.num_heads = num_heads

        self.W_q = nn.Parameter(torch.rand(d_in, d_out_kq))
        self.W_k = nn.Parameter(torch.rand(d_in, d_out_kq))
        self.W_v = nn.Parameter(torch.rand(d_in, d_out_v))
        self.Wo = nn.Parameter(torch.rand(d_out_v,d_in)) # Each head gathers information independently. Wo learns how to turn those separate outputs into a coordinated representation.

    def forward(self, x):
        queries = x @ self.W_q # (batch_size, sequence_length, d_out_kq)
        keys = x @ self.W_k # (batch_size, sequence_length, d_out_kq)
        values = x @ self.W_v # (batch_size, sequence_length, d_out_v) # each of the 2 sequences has 4 tokens, and each token is now represented by a d_out_v dim vector. (24 value features if d_out_v = 24)

        # (2, 4, 6) 2 sequences, each sequence has 4 tokens, and each token is represented by 6-dim vector.
        batch_size, sequence_length, _ = x.shape

        #Split into heads
        head_dim_kq = self.d_out_kq // self.num_heads 
        head_dim_v = self.d_out_v // self.num_heads

        #reshape
        queries = queries.reshape(batch_size, sequence_length, self.num_heads, head_dim_kq) # 2 sequences, each sequence has 4 tokens, each token has 3 query vectors (one from each head), and each query vector has dim head_dim_kq
        keys = keys.reshape(batch_size, sequence_length, self.num_heads, head_dim_kq) # 2 sequences, each sequence has 4 tokens, each token has 3 key vectors (one from each head), and each key vector has dim head_dim_kq
        values = values.reshape(batch_size, sequence_length, self.num_heads, head_dim_v) # 2 sequences, each sequence has 4 tokens, each token as 3 value vectors (one from each head), and each value vector has dim head_dim_v

        #transpose
        queries = queries.transpose(1,2) # (batch_size, num_heads, sequence_length, head_dim_kq) # 2 sequences, each sequence has 3 heads, each head has 4 tokens, each token is represented by dim head_dim_kq
        keys = keys.transpose(1,2)  # (batch_size, num_heads, sequence_length, head_dim_kq)
        values = values.transpose(1,2) # (batch_size, num_heads, sequence_length, head_dim_v) 

        #pytorch performs the batched multiplication
        attention_scores = queries @ keys.transpose(-2, -1) # (batch_size, num_heads, sequence_length, sequence_length)

        #without diagonal=1, the diagonal is also blocked. 
        mask = torch.triu(torch.ones(sequence_length, sequence_length), diagonal=1)
        masked = attention_scores.masked_fill(mask.bool(), -torch.inf)

        attention_weights = torch.softmax(masked / head_dim_kq ** 0.5, dim=-1)

        context_vec = attention_weights @ values
        context_vec = context_vec.transpose(1,2) # (batch_size, num_heads, sequence_length, head_dim_v) -> (batch_size, sequence_length, num_heads, head_dim_v)
        context_vec = context_vec.reshape(batch_size, sequence_length, self.num_heads * head_dim_v)

        # Wo maps the concatenated output back to d_in dimensions.
        context_vec = context_vec @ self.Wo
        return context_vec



